# einops-reduce — worked example 2: Perform 3x3 max-pooling with einops.reduce using axis decomposition

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-reduce`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import torch.nn.functional as F
import einops
from einops import reduce

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The `'b c (h p1) (w p2) -> b c h w'` pattern partitions the spatial axes into non-overlapping patches of size `(p1, p2)`. Providing `'max'` as the reduction op and setting `p1=3, p2=3` takes the maximum within each `3x3` patch. This is identical to `F.max_pool2d(x, kernel_size=3, stride=3)` expressed declaratively.

## Worked solution

Input: `(B=1, C=2, H=6, W=6)`. We want `(B, C, 2, 2)` — a 3x3 max pool.

**Pattern:** `'b c (h p1) (w p2) -> b c h w'` with `p1=3, p2=3`.

**Step 1.** Einops splits `H=6` into `h=2` groups of `p1=3` and `W=6` into `w=2` groups of `p2=3`.
**Step 2.** For each `(b, c, h, w)` output cell, it takes the max over the corresponding `3x3` patch in the input.
**Step 3.** Output shape: `(1, 2, 2, 2)`.

The key: the patch axes `p1, p2` appear on the left but not the right, so they are reduced. The block axes `h, w` appear on both sides and are preserved.

In [ ]:
import torch as t
from einops import reduce

t.manual_seed(22)
B, C, H, W = 2, 3, 9, 9
x = t.randn(B, C, H, W)

def max_pool_3x3(x):
    return reduce(x, 'b c (h p1) (w p2) -> b c h w', 'max', p1=3, p2=3)

out = max_pool_3x3(x)
print('Input shape:', x.shape)   # (2, 3, 9, 9)
print('Output shape:', out.shape)  # (2, 3, 3, 3)
assert out.shape == (B, C, H // 3, W // 3)

# Verify against PyTorch F.max_pool2d
import torch.nn.functional as F
ref = F.max_pool2d(x, kernel_size=3, stride=3)
assert t.allclose(out, ref)
print('Matches F.max_pool2d(kernel_size=3, stride=3):', True)